# 01 — Korpus & Annotation

Phase 1 (Korpus + Bias-Notiz) und Phase 2 (κ + drei Edge Cases) leben in diesem Notebook. Pipeline → `02_extract.ipynb`, Eval → `03_eval.ipynb`, Frontier → `04_frontier_compare.ipynb`.

## Run-Header

| Feld | Wert |
|---|---|
| Datum (Phase 1) | _YYYY-MM-DD_ |
| Datum (Phase 2) | _YYYY-MM-DD_ |
| Korpus-Datei | `daten/eigener_korpus.jsonl` |
| Anzahl Anzeigen im Korpus | _ |
| Genutzte Suchanfragen (Phase 1) | _ |
| Pair-Partner:in (Phase 2) | _ |
| 12 gemeinsame Anzeigen-IDs | _ |

## Phase 1 — Korpus inspizieren + Bias-Notiz

In [1]:
import json
from collections import Counter

with open("../daten/eigener_korpus.jsonl", encoding="utf-8") as f:
    anzeigen = [json.loads(line) for line in f]

print(f"Total: {len(anzeigen)}")
print("\nAngebotsart-Verteilung:")
for art, n in Counter(a["angebotsart"] for a in anzeigen).most_common():
    print(f"  {art}: {n}")

print("\nVertragsdauer-Verteilung:")
for vd, n in Counter(a["vertragsdauer"] for a in anzeigen).most_common():
    print(f"  {vd}: {n}")

print(f"\nDurchschnittliche Textlänge: "
      f"{sum(len(a['text']) for a in anzeigen) // len(anzeigen)} Zeichen")
print(f"Min/Max Textlänge: "
      f"{min(len(a['text']) for a in anzeigen)} / "
      f"{max(len(a['text']) for a in anzeigen)}")

Total: 55

Angebotsart-Verteilung:
  ARBEIT: 35
  AUSBILDUNG: 20

Vertragsdauer-Verteilung:
  KEINE_ANGABE: 33
  UNBEFRISTET: 19
  BEFRISTET: 3

Durchschnittliche Textlänge: 2933 Zeichen
Min/Max Textlänge: 497 / 6917


## Bias-Notiz zum Korpus

**Quelle:** Arbeitsagentur Jobsuche-API (öffentliche REST-API, Stand 2026-05-11).

**Verzerrungen, die unsere Pipeline beeinflussen können:**

- **Branchen-Bias:** Alle Queries zielen auf Daten-/IT-nahe Rollen (Data Scientist, 
  Data Engineer, Fachinformatiker, BI). Das Schema-Feld `skills_top3` wird daher 
  überproportional Python/SQL/ML-Tools enthalten — andere technische Skills 
  (z.B. Lab-Geräte, CAD, ERP) sind unterrepräsentiert.

- **Vertragsart-Verteilung:** 35 Festanstellungen + 20 Ausbildungen, aber 
  **0 Praktika und 0 Werkstudent-Stellen**. Die Praktikum-Queries 
  (`angebotsart=34`) lieferten keine Treffer. Konsequenz für Phase 2: bei 
  unserer 12er-Stichprobe kommen die Schema-Werte `praktikum` und `werkstudent` 
  vermutlich nicht vor — Modell-Performance auf diesen Klassen ist in Phase 3 
  nicht evaluierbar.

- **Plattform-Bias:** Externe Anbieter (Präfixe `11949-`, `12634-`, `12288-`, 
  `14225-`) wurden ausgeschlossen, weil dort der Volltext fehlt. Damit fehlen 
  z.B. Stellen großer Job-Aggregatoren.

- **Region:** Bundesweit gesammelt, aber mit gezielten Queries für München und 
  Berlin → leichter Bias zu Großstädten.

- **Sprachlich:** Texte enthalten Markdown-Syntax (`**Überschrift**`, `* Liste`) 
  als Strukturmarker. Das hilft Qwen2.5 beim Verstehen, ist aber kein "natürliches" 
  Textformat.

**Konsequenz für Phase 2:** Bei der 12er-Stichprobe ziehen wir bewusst gemischt — 
ein paar Ausbildungen, ein paar Festanstellungen — damit Cohen's κ überhaupt 
Disagreement-Potenzial hat.

## Phase 2 — κ-Tabelle + drei Edge Cases

In [2]:
"""
Phase 2 — 12er-Stichprobe für Hand-Annotation reproduzierbar ziehen.
Gemischt aus ARBEIT + AUSBILDUNG, damit Schema-Felder Vielfalt zeigen.
"""
import json
import random
from pathlib import Path
from collections import Counter

random.seed(42)  # reproduzierbar — Partner:in zieht mit demselben Seed dieselben Anzeigen

# Korpus laden
korpus_pfad = Path("../daten/eigener_korpus.jsonl")
anzeigen = [json.loads(line) for line in korpus_pfad.open(encoding="utf-8")]
print(f"Korpus geladen: {len(anzeigen)} Anzeigen")

# Aufteilen nach angebotsart
arbeit     = [a for a in anzeigen if a["angebotsart"] == "ARBEIT"]
ausbildung = [a for a in anzeigen if a["angebotsart"] == "AUSBILDUNG"]
print(f"  davon ARBEIT: {len(arbeit)}, AUSBILDUNG: {len(ausbildung)}")

# 8 Arbeit + 4 Ausbildung = 12
N_ARBEIT, N_AUSBILDUNG = 8, 4
stichprobe = random.sample(arbeit, N_ARBEIT) + random.sample(ausbildung, N_AUSBILDUNG)
random.shuffle(stichprobe)  # mischen, damit Reihenfolge nicht nach Kategorie sortiert ist

print(f"\nStichprobe: {len(stichprobe)} Anzeigen")
print(f"  Verteilung: {Counter(a['angebotsart'] for a in stichprobe)}")

print("\nRefnrs für meine_gold.csv + Partner-CSV:")
for i, a in enumerate(stichprobe, 1):
    print(f"  {i:>2}. {a['refnr']:<20} | {a['angebotsart']:<10} | {a['titel'][:55]}")

Korpus geladen: 55 Anzeigen
  davon ARBEIT: 35, AUSBILDUNG: 20

Stichprobe: 12 Anzeigen
  Verteilung: Counter({'ARBEIT': 8, 'AUSBILDUNG': 4})

Refnrs für meine_gold.csv + Partner-CSV:
   1. 13644-294294-S       | ARBEIT     | Data Scientist (m/w/d)
   2. 10001-1002678126-S   | ARBEIT     | Data Scientist (m/w/d)
   3. 10000-1185445644-S   | ARBEIT     | Data Scientist (m/w/d)
   4. 10001-1002026714-S   | AUSBILDUNG | Fachinformatiker - Anwendungsentwicklung (m/w/d)
   5. 15719-44156922-55-S  | ARBEIT     | Data Scientist in Advanced Analytics (m/w/d)
   6. 16818-100787927-S    | AUSBILDUNG | Fachinformatiker*in Anwendungsentwicklung
   7. 19384-938440348-S    | ARBEIT     | Data Scientist (m/w/d) – Pharmagroßhandel – Standort Es
   8. 10001-1003030981-S   | AUSBILDUNG | Fachinformatiker/in - Anwendungsentwicklung
   9. 16818-100814418-S    | AUSBILDUNG | Fachinformatiker*in Anwendungsentwicklung
  10. 13644-297872-S       | ARBEIT     | Data Scientist (m/w/d)
  11. 13319-868490/1_60741

In [4]:
"""
Helfer: Anzeigentext nach refnr anzeigen — bequem fürs Annotieren.
"""
# Lookup-Dict bauen
nach_refnr = {a["refnr"]: a for a in anzeigen}

def zeige(refnr_oder_index):
    """Akzeptiert refnr-String ODER Index (1-12) aus der Stichprobe."""
    if isinstance(refnr_oder_index, int):
        refnr = stichprobe[refnr_oder_index - 1]["refnr"]   # FIX: ["refnr"] holt den String aus dem Dict
    else:
        refnr = refnr_oder_index
    a = nach_refnr[refnr]
    print(f"═══ refnr: {a['refnr']} ═══")
    print(f"Titel:  {a['titel']}")
    print(f"Firma:  {a['firma']}")
    print(f"Art:    {a['angebotsart']} | Vertragsdauer: {a['vertragsdauer']}")
    print(f"Ort:    {a['ort_ort']} ({a['ort_region']})")
    print(f"\n{'─'*70}\n")
    print(a['text'])
    print(f"\n{'─'*70}")
    print(f"Schema-Check-Hilfe:")
    print(f"  homeoffice:      ja / teilweise / nein / remote / nicht_genannt")
    print(f"  vertragsart:     ausbildung / festanstellung / praktikum / werkstudent / sonstiges")
    print(f"  erfahrungslevel: junior / mid / senior / egal / nicht_genannt")

# Benutzung: zeige(1) für die erste Anzeige, zeige(2) für die zweite, etc.
zeige(12)

═══ refnr: 10001-1002616378-S ═══
Titel:  Data Scientist (m/w/d)
Firma:  team SE
Art:    ARBEIT | Vertragsdauer: UNBEFRISTET
Ort:    Flensburg (Schleswig-Holstein)

──────────────────────────────────────────────────────────────────────

Die DLG Gruppe zählt zu den führenden Unternehmen Europas in den Bereichen Lebensmittel, Energie und Baustoffe. Als Genossenschaft im Besitz von 25.000 dänischen Landwirten hat sie ihren Hauptsitz in Fredericia, Dänemark. Unsere Wurzeln sind dänisch, doch unser Blick ist global. Ausgehend von einer rein dänischen Tätigkeit hat sich die DLG Gruppe in den letzten 20 Jahren erheblich vergrößert und ist inzwischen in 18 Ländern aktiv.

Die team SE ist ein wichtiger Bestandteil der DLG Gruppe. In team SE übernehmen unsere rund 350 Mitarbeitenden übergeordnete Servicefunktionen für die gesamte Gruppe. Wir bieten vielseitige und abwechslungsreiche Aufgaben in einem soliden, wachsenden Unternehmen mit mittelständischer Tradition und modernem Arbeitsumfeld.

###

## Phase 2 — κ-Tabelle + drei Edge Cases

### Methodischer Hinweis

Aufgrund organisatorischer Schwierigkeiten beim ursprünglichen Pair-Matching wurde 
die zweite Annotation (`partner_gold.csv`) nach Rücksprache mit dem Lehrer von einem 
KI-Assistenten (Claude, Anthropic) unabhängig durchgeführt. Methodische 
Einschränkung: Cohen's κ misst hier nicht klassisches Inter-Annotator-Agreement 
zwischen zwei Menschen, sondern Mensch-vs-LLM-Agreement. Die Edge Cases bleiben 
inhaltlich aussagekräftig für die Schema-Diskussion.

### Cohen's κ über 12 Anzeigen

| Feld             | κ      | Übereinst. | Interpretation (Landis & Koch) |
|------------------|--------|------------|--------------------------------|
| homeoffice       | 1.000  | 100 %      | fast perfekt                   |
| vertragsart      | 1.000  | 100 %      | fast perfekt                   |
| erfahrungslevel  | 0.625  | 75 %       | substanziell                   |

### Interpretation der Werte

**`homeoffice` und `vertragsart` (κ = 1.0):** Volle Übereinstimmung. Dieser 
hohe Wert ist nicht primär ein Indiz für ein perfekt definiertes Schema, sondern 
durch die **Korpus-Zusammensetzung** beeinflusst:
- Bei `homeoffice` waren 8 von 12 Anzeigen schlicht stumm zum Thema → triviales 
  `nicht_genannt`. Die 3 `teilweise`-Fälle hatten sehr eindeutige Formulierungen 
  ("hybrid", "3 Tage HO/Woche", "Home Office möglich"). 
- Bei `vertragsart` ist die Unterscheidung Ausbildung vs. Festanstellung im Korpus 
  textuell eindeutig (Wort "Ausbildung" / "Auszubildende" steht explizit drin).
- Mit nur 12 Items ist κ = 1.0 statistisch fragil — bereits **ein** Disagreement 
  hätte den Wert deutlich unter 1.0 gedrückt.

**`erfahrungslevel` (κ = 0.625):** Substanziell, aber mit 3 Disagreements. Das 
ist der **inhaltlich aussagekräftigste Wert** und zeigt eine reale Schema-Lücke 
— siehe Edge Cases unten.

### Drei Edge Cases (alle bei `erfahrungslevel`)

#### Edge Case 1 — KOSATEC Data Scientist (refnr 13644-294294-S)
- **Annotator A (Mensch):** `nicht_genannt`
- **Annotator B (Claude):** `mid`
- **Text-Signal:** *"Praktische Erfahrungen mit Datenbanken"*, *"Idealerweise erste 
  Erfahrungen mit BI-Tools"* — keine Jahresangabe, kein Titel-Signal.
- **Schema-Lücke:** Wenn Erfahrung erwähnt wird, aber **weder Jahresangabe noch 
  Titel-Marker** vorhanden ist, ist unklar, ob das als "nicht_genannt" (streng) 
  oder als "mid" (wohlwollend) zu werten ist. Annotator A las streng nach 
  Schema-Definition ("≤2 / 2–5 / ≥5 Jahre"), Annotator B interpretierte das 
  Vorhandensein von "praktische Erfahrung" als implizites Level-Signal.

#### Edge Case 2 — Lantenhammer Consulting (refnr 10000-1185445644-S)
- **Annotator A (Mensch):** `senior`
- **Annotator B (Claude):** `mid`
- **Text-Signal:** *"Mindestens 2 – 3 Jahre Berufserfahrung"*, dazu 
  *"Erfahrung in der Leitung kleiner, agiler Projekte"* und Anforderung 
  *"M.Sc. oder Ph.D."*.
- **Schema-Lücke:** Die Jahresangabe "2-3 Jahre" liegt exakt in der `mid`-Range 
  des Schemas (2-5 Jahre). Annotator B blieb streng bei der Zahl. Annotator A 
  gewichtete zusätzliche Senior-Marker (Führungsaspekt, hohe formale 
  Anforderungen) höher und entschied `senior` trotz fehlendem expliziten 
  Senior-Titel. Schema definiert nicht, wie Sekundär-Signale gegen primäre 
  Jahresangabe gewichtet werden.

#### Edge Case 3 — Allianz Advanced Analytics (refnr 15719-44156922-55-S)
- **Annotator A (Mensch):** `nicht_genannt`
- **Annotator B (Claude):** `mid`
- **Text-Signal:** *"Praxiserfahrung"*, *"routiniert in Python"*, *"tiefgehendes 
  Verständnis"* — keine Jahresangabe, kein Titel-Signal.
- **Schema-Lücke:** Adjektive wie "routiniert" und "tiefgehend" sind nicht in den 
  Schema-Definitionen abgebildet. Annotator A wertete sie als nicht-quantifizierbar 
  und entschied `nicht_genannt`. Annotator B wertete sie als implizites 
  Mittel-Level-Signal.

### Schema-Empfehlungen aus den Edge Cases

1. **Erfahrungslevel-Schema sollte einen "implizit erfahren, aber ohne Zahl/Titel"-Wert 
   bekommen** — z.B. eine eigene Kategorie zwischen `nicht_genannt` und `mid`, oder 
   eine klare Regel: "Adjektive wie 'routiniert', 'praktisch', 'tiefgehend' → mid".

2. **Sekundär-Marker (Führungsaufgaben, M.Sc.-Anforderung) sollten explizit 
   gewichtet werden** — entweder als Senior-Indikator definieren oder explizit 
   ausschließen.

3. **Konvention für Range-Angaben** ("Mindestens 2-3 Jahre"): Untere Grenze 
   nehmen (analog zu `gehalt_min_eur`) oder beim Mittelwert orientieren?